# Notebook 2: Multiple Dosing and Steady-State Concentration: From Accumulation to Sustained Efficacy

This notebook is the second foundational module of the Clinical Pharmacy PK/PD Interactive Simulation Platform.

In Notebook 1, we learned how plasma drug concentration changes over time after a single dose, and how plasma concentration can be further translated into drug effect.

This section further discusses a more common clinical situation: **multiple dosing**.

Clinical drug therapy rarely involves only a single dose. Most drugs need to be administered repeatedly at a fixed dosing interval, for example:

- q6h: once every 6 hours
- q8h: once every 8 hours
- q12h: once every 12 hours
- q24h: once every 24 hours

After multiple dosing, drug in the body may gradually accumulate and eventually approach steady-state concentration.

The core logic of this notebook is:

$$
Dose + \tau \rightarrow Accumulation \rightarrow Steady\ state \rightarrow Efficacy/Safety
$$

where $\tau$ represents the dosing interval.

This section still uses a one-compartment model for teaching simulation and considers:

- IV bolus dosing
- Oral dosing
- Accumulation after multiple dosing
- Steady-state concentration
- Peak and trough concentrations
- Therapeutic window
- Emax pharmacodynamic model


## 1. Learning Objectives

After completing this notebook, you should be able to:

1. Explain why drug accumulation occurs during multiple dosing.
2. Describe how dosing interval, half-life, clearance, and volume of distribution affect concentration-time profiles.
3. Interpret steady state and explain why it is usually approached after approximately 4–5 half-lives.
4. Compare IV bolus and oral multiple-dosing profiles using peak, trough, and average concentrations.
5. Use the therapeutic window and Emax model to evaluate efficacy and safety during repeated dosing.


## 2. Basic Concepts of Multiple Dosing

Multiple dosing means administering a drug repeatedly at fixed time intervals.

Assume the dosing interval is:

$$
\tau
$$

If the first dose is given at time 0, subsequent dosing times can be expressed as:

$$
0, \tau, 2\tau, 3\tau, \cdots
$$

Each dose produces a new concentration-time curve. The total concentration after multiple dosing can be viewed as the superposition of the concentration contributions from each dose.

If the previous dose has not been completely eliminated before the next dose is given, the amount of drug in the body gradually increases. This phenomenon is called:

$$
Accumulation
$$

Drug accumulation.

The degree of drug accumulation is mainly affected by the following factors:

- Half-life $t_{1/2}$
- Dosing interval $\tau$
- Clearance CL
- Volume of distribution Vd
- Dose

In general, the longer the half-life and the shorter the dosing interval, the more likely the drug is to accumulate.


## 3. One-Compartment Model for IV Bolus Multiple Dosing

For IV bolus dosing, the plasma concentration after a single dose is:

$$
C(t) = \frac{Dose}{V_d} \cdot e^{-kt}
$$

where:

$$
k = \frac{CL}{V_d}
$$

During multiple dosing, the total concentration is the superposition of the concentrations after each dose:

$$
C(t) = \sum_{i=0}^{n-1} \frac{Dose}{V_d} \cdot e^{-k(t-i\tau)}
$$

with the condition:

$$
t \geq i\tau
$$

In other words, only doses that have already been administered contribute to the current concentration.

After steady state is reached, the steady-state peak concentration after IV bolus dosing can be expressed as:

$$
C_{max,ss} = \frac{Dose/V_d}{1-e^{-k\tau}}
$$

The steady-state trough concentration can be expressed as:

$$
C_{min,ss} = C_{max,ss} \cdot e^{-k\tau}
$$

The accumulation factor is:

$$
R = \frac{1}{1-e^{-k\tau}}
$$

A larger $R$ indicates more pronounced drug accumulation.


## 4. One-Compartment Model for Oral Multiple Dosing

For oral dosing with first-order absorption and first-order elimination, the plasma concentration after a single dose is:

$$
C(t) = \frac{F \cdot Dose \cdot k_a}{V_d(k_a-k)}
\left(e^{-kt} - e^{-k_a t}\right)
$$

where:

$$
k = \frac{CL}{V_d}
$$

After multiple oral doses, the total concentration can also be viewed as the superposition of the concentration contributions from each dose:

$$
C(t) = \sum_{i=0}^{n-1}
\frac{F \cdot Dose \cdot k_a}{V_d(k_a-k)}
\left(e^{-k(t-i\tau)} - e^{-k_a(t-i\tau)}\right)
$$

with the condition:

$$
t \geq i\tau
$$

Compared with IV bolus multiple dosing, oral multiple dosing has the following characteristics:

- After each dose, concentration first rises and then falls.
- Tmax may occur within each dosing interval.
- ka affects the time at which the peak concentration appears.
- F affects overall exposure.
- After repeated dosing, concentration also gradually approaches steady state.


## 5. Basic Meaning of Steady-State Concentration

Steady state does not mean that plasma drug concentration no longer fluctuates.

For intermittent dosing, such as q12h dosing, steady state means:

> The shape of the concentration curve within each dosing interval is essentially repeated.

In other words, at steady state:

- The peak concentration after each dose is essentially the same.
- The trough concentration before the next dose is essentially the same.
- The AUC within each dosing interval is essentially the same.

The steady-state average concentration can be expressed as:

For IV dosing:

$$
C_{ss,avg} = \frac{Dose}{CL \cdot \tau}
$$

For oral dosing:

$$
C_{ss,avg} = \frac{F \cdot Dose}{CL \cdot \tau}
$$

The formulas show that:

- Increasing Dose increases steady-state average concentration.
- Increasing CL decreases steady-state average concentration.
- Shortening $\tau$ increases steady-state average concentration.
- During oral dosing, decreasing F decreases steady-state average concentration.


## 6. Why Does It Usually Take 4–5 Half-Lives to Approach Steady State?

For drugs with first-order elimination, the rate of approaching steady state is mainly determined by half-life, not by dose size.

After different numbers of half-lives, the approximate fraction of steady state reached is:

| Time | Fraction of steady state reached |
|---|---|
| 1 half-life | 50% |
| 2 half-lives | 75% |
| 3 half-lives | 87.5% |
| 4 half-lives | 93.75% |
| 5 half-lives | 96.875% |

Therefore, in clinical practice it is often stated that:

> After approximately 4–5 half-lives, drug concentration usually approaches steady state.

Note that:

- Increasing the dose can raise the steady-state concentration level.
- However, increasing the dose usually does not make the drug reach steady state faster.
- The rate of reaching steady state is mainly determined by half-life.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


def one_compartment_iv_bolus_single(t_after_dose, dose_mg, vd_l, cl_l_h):
    """
    Single-dose contribution after IV bolus dosing.
    """
    k_elim = cl_l_h / vd_l
    concentration = (dose_mg / vd_l) * np.exp(-k_elim * t_after_dose)
    return concentration


def one_compartment_oral_single(t_after_dose, dose_mg, vd_l, cl_l_h, ka_h, bioavailability):
    """
    Single-dose contribution after oral dosing with first-order absorption.
    """
    k_elim = cl_l_h / vd_l

    if np.isclose(ka_h, k_elim):
        concentration = (
            bioavailability
            * dose_mg
            / vd_l
            * k_elim
            * t_after_dose
            * np.exp(-k_elim * t_after_dose)
        )
    else:
        concentration = (
            bioavailability
            * dose_mg
            * ka_h
            / (vd_l * (ka_h - k_elim))
            * (np.exp(-k_elim * t_after_dose) - np.exp(-ka_h * t_after_dose))
        )

    return np.maximum(concentration, 0)


def simulate_multiple_dosing(
    t,
    route,
    dose_mg,
    vd_l,
    cl_l_h,
    tau_h,
    n_doses,
    ka_h=1.2,
    bioavailability=0.8
):
    """
    Simulate multiple dosing using superposition.
    """
    concentration = np.zeros_like(t, dtype=float)
    dose_times = np.arange(n_doses) * tau_h

    for dose_time in dose_times:
        mask = t >= dose_time
        t_after_dose = t[mask] - dose_time

        if route == "IV bolus":
            concentration[mask] += one_compartment_iv_bolus_single(
                t_after_dose=t_after_dose,
                dose_mg=dose_mg,
                vd_l=vd_l,
                cl_l_h=cl_l_h
            )
        else:
            concentration[mask] += one_compartment_oral_single(
                t_after_dose=t_after_dose,
                dose_mg=dose_mg,
                vd_l=vd_l,
                cl_l_h=cl_l_h,
                ka_h=ka_h,
                bioavailability=bioavailability
            )

    return concentration, dose_times


def calculate_interval_metrics(t, concentration, interval_start, interval_end):
    """
    Calculate Cmax, Tmax, Cmin, AUC, and average concentration within an interval.
    """
    mask = (t >= interval_start) & (t <= interval_end)
    t_interval = t[mask]
    c_interval = concentration[mask]

    if len(t_interval) < 2:
        return {
            "Cmax": np.nan,
            "Tmax": np.nan,
            "Cmin": np.nan,
            "Tmin": np.nan,
            "AUC_interval": np.nan,
            "Cavg_interval": np.nan
        }

    cmax = np.max(c_interval)
    tmax = t_interval[np.argmax(c_interval)]
    cmin = np.min(c_interval)
    tmin = t_interval[np.argmin(c_interval)]
    auc_interval = np.trapz(c_interval, t_interval)
    cavg_interval = auc_interval / (interval_end - interval_start)

    return {
        "Cmax": cmax,
        "Tmax": tmax,
        "Cmin": cmin,
        "Tmin": tmin,
        "AUC_interval": auc_interval,
        "Cavg_interval": cavg_interval
    }


def calculate_basic_parameters(route, dose_mg, vd_l, cl_l_h, tau_h, ka_h, bioavailability):
    """
    Calculate basic PK parameters.
    """
    k_elim = cl_l_h / vd_l
    half_life = np.log(2) / k_elim

    if route == "IV bolus":
        dose_factor = 1.0
    else:
        dose_factor = bioavailability

    theoretical_cavg_ss = dose_factor * dose_mg / (cl_l_h * tau_h)
    theoretical_auc_tau_ss = dose_factor * dose_mg / cl_l_h
    accumulation_factor_iv = 1 / (1 - np.exp(-k_elim * tau_h))
    time_to_90_ss = -np.log(1 - 0.90) / k_elim
    time_to_95_ss = -np.log(1 - 0.95) / k_elim

    return {
        "k_elim": k_elim,
        "Half-life": half_life,
        "Theoretical Cavg,ss": theoretical_cavg_ss,
        "Theoretical AUC_tau,ss": theoretical_auc_tau_ss,
        "Accumulation factor IV": accumulation_factor_iv,
        "Time to 90% SS": time_to_90_ss,
        "Time to 95% SS": time_to_95_ss
    }

## 7. Interactive Simulation 1: Concentration-Time Curve After Multiple Dosing

The simulation below allows you to choose between two routes of administration:

- IV bolus
- Oral

Please observe:

- As the number of doses increases, does the concentration gradually rise?
- Do the curves after later doses gradually approach a repeated pattern?
- When the dosing interval is shortened, does accumulation become more pronounced?
- When clearance decreases, does the concentration rise more easily?
- How do the curve shapes differ between oral dosing and IV dosing?


In [ ]:
def plot_multiple_dosing_profile(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    tau_h=12,
    n_doses=8,
    ka_h=1.2,
    bioavailability=0.8
):
    t_end_h = n_doses * tau_h
    t = np.linspace(0, t_end_h, 2000)

    concentration, dose_times = simulate_multiple_dosing(
        t=t,
        route=route,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        n_doses=n_doses,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    params = calculate_basic_parameters(
        route=route,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    last_interval_start = (n_doses - 1) * tau_h
    last_interval_end = n_doses * tau_h
    last_metrics = calculate_interval_metrics(
        t=t,
        concentration=concentration,
        interval_start=last_interval_start,
        interval_end=last_interval_end
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, concentration, linewidth=2, label=f"{route}")

    for dose_time in dose_times:
        ax.axvline(dose_time, linestyle=":", alpha=0.4)

    ax.set_title("Multiple Dosing Concentration-Time Profile")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Parameter": [
            "Route",
            "Dose",
            "Dosing interval tau",
            "Number of doses",
            "Vd",
            "CL",
            "ka",
            "Bioavailability",
            "Elimination rate constant",
            "Half-life",
            "Theoretical Cavg,ss",
            "Time to 90% steady state",
            "Time to 95% steady state",
            "Last-interval Cmax",
            "Last-interval Cmin",
            "Last-interval AUC",
            "Last-interval average concentration"
        ],
        "Value": [
            route,
            f"{dose_mg:.1f} mg",
            f"{tau_h:.1f} h",
            f"{n_doses}",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.1f} L/h",
            f"{ka_h:.2f} 1/h",
            f"{bioavailability:.2f}",
            f"{params['k_elim']:.4f} 1/h",
            f"{params['Half-life']:.2f} h",
            f"{params['Theoretical Cavg,ss']:.2f} mg/L",
            f"{params['Time to 90% SS']:.2f} h",
            f"{params['Time to 95% SS']:.2f} h",
            f"{last_metrics['Cmax']:.2f} mg/L",
            f"{last_metrics['Cmin']:.2f} mg/L",
            f"{last_metrics['AUC_interval']:.2f} mg*h/L",
            f"{last_metrics['Cavg_interval']:.2f} mg/L"
        ]
    })

    display(summary)


interact(
    plot_multiple_dosing_profile,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    tau_h=FloatSlider(value=12, min=4, max=48, step=2, description="Tau"),
    n_doses=IntSlider(value=8, min=1, max=20, step=1, description="Doses"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F")
);

## 8. Observation Task 1: Multiple-Dose Accumulation

Please complete the following operations:

### Task A: Standard oral multiple dosing

Set:

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- tau = 12 h
- Doses = 8
- ka = 1.2 1/h
- F = 0.8

Observe:

- Do the concentration peaks after the first few doses gradually increase?
- Do the curves during the last few dosing intervals tend to repeat?
- What do Last-interval Cmax and Last-interval Cmin represent?

### Task B: Shorten the dosing interval

Change tau from 12 h to 6 h.

Observe:

- Does the concentration accumulate more easily?
- Does Last-interval Cmin increase?
- Does Theoretical Cavg,ss increase?

### Task C: Reduce clearance

Change CL from 5 L/h to 2 L/h.

Observe:

- Is the half-life prolonged?
- Is the time required to approach steady state prolonged?
- Is the concentration after multiple dosing higher?


## 9. Peak Concentration, Trough Concentration, and Steady-State Fluctuation

During multiple intermittent dosing, concentration usually fluctuates up and down within each dosing interval.

Common metrics include:

| Metric | Meaning |
|---|---|
| Cmax | Peak concentration, the highest concentration within a dosing interval |
| Cmin | Trough concentration, the lowest concentration before the next dose or within a dosing interval |
| Cavg | Average concentration, the average exposure level within a dosing interval |
| AUCτ | Area under the concentration-time curve within one dosing interval |

For clinical pharmacy, these metrics are all important:

- Excessively high Cmax may be associated with toxicity for some drugs.
- Excessively low Cmin may lead to insufficient efficacy.
- Cavg and AUC reflect overall exposure.
- Peak-to-trough fluctuation at steady state is jointly affected by Dose, CL, Vd, and $\tau$.

Different drugs focus on different metrics. For example:

- Aminoglycosides place more emphasis on Cmax/MIC.
- Vancomycin places more emphasis on AUC/MIC.
- Some narrow-therapeutic-window drugs place more emphasis on trough concentration or the steady-state concentration range.


## 10. Interactive Simulation 2: Observing the Last Dosing Interval

When judging whether concentration has approached steady state, it is often necessary to zoom in on the last dosing interval.

The simulation below displays:

- The complete multiple-dose concentration curve
- The concentration curve within the last dosing interval
- Cmax, Cmin, and Cavg within the last dosing interval

Please observe how peak-to-trough fluctuation changes after the dosing interval $\tau$ is changed.


In [ ]:
def plot_last_dosing_interval(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    tau_h=12,
    n_doses=10,
    ka_h=1.2,
    bioavailability=0.8
):
    t_end_h = n_doses * tau_h
    t = np.linspace(0, t_end_h, 3000)

    concentration, dose_times = simulate_multiple_dosing(
        t=t,
        route=route,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        n_doses=n_doses,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    last_interval_start = (n_doses - 1) * tau_h
    last_interval_end = n_doses * tau_h

    mask = (t >= last_interval_start) & (t <= last_interval_end)
    t_last = t[mask]
    c_last = concentration[mask]

    metrics = calculate_interval_metrics(
        t=t,
        concentration=concentration,
        interval_start=last_interval_start,
        interval_end=last_interval_end
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, concentration, linewidth=2, label="Full profile")
    ax.axvspan(last_interval_start, last_interval_end, alpha=0.15, label="Last interval")
    ax.set_title("Full Multiple Dosing Profile")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(t_last, c_last, linewidth=2, label="Last interval")
    ax.axhline(metrics["Cavg_interval"], linestyle="--", label=f"Cavg = {metrics['Cavg_interval']:.2f} mg/L")
    ax.scatter(metrics["Tmax"], metrics["Cmax"], zorder=5, label=f"Cmax = {metrics['Cmax']:.2f}")
    ax.scatter(metrics["Tmin"], metrics["Cmin"], zorder=5, label=f"Cmin = {metrics['Cmin']:.2f}")
    ax.set_title("Last Dosing Interval")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Last-interval start",
            "Last-interval end",
            "Cmax",
            "Time of Cmax",
            "Cmin",
            "Time of Cmin",
            "AUC_tau",
            "Cavg"
        ],
        "Value": [
            f"{last_interval_start:.2f} h",
            f"{last_interval_end:.2f} h",
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Tmax']:.2f} h",
            f"{metrics['Cmin']:.2f} mg/L",
            f"{metrics['Tmin']:.2f} h",
            f"{metrics['AUC_interval']:.2f} mg*h/L",
            f"{metrics['Cavg_interval']:.2f} mg/L"
        ]
    })

    display(summary)


interact(
    plot_last_dosing_interval,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    tau_h=FloatSlider(value=12, min=4, max=48, step=2, description="Tau"),
    n_doses=IntSlider(value=10, min=2, max=24, step=1, description="Doses"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F")
);

## 11. Observation Task 2: Dosing Interval and Peak-to-Trough Fluctuation

Please use Interactive Simulation 2 to complete the following tasks.

### Task A: Standard regimen

Set:

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- tau = 12 h
- Doses = 10
- ka = 1.2 1/h
- F = 0.8

Record:

- Cmax
- Cmin
- Cavg
- AUCτ

### Task B: Shorten the dosing interval

Change tau to 6 h.

Observe:

- Does Cmin increase noticeably?
- Does Cavg increase?
- Does peak-to-trough fluctuation become smaller?

### Task C: Extend the dosing interval

Change tau to 24 h.

Observe:

- Does Cmin decrease?
- Could the risk of concentration falling below the effective range increase?
- Does peak-to-trough fluctuation become larger?


## 12. Therapeutic Window: Insufficient Efficacy and Toxicity Risk During Multiple Dosing

Therapeutic-window analysis is very important in multiple dosing.

This section continues to use two simplified metrics:

| Concept | Meaning |
|---|---|
| MEC | Minimum Effective Concentration |
| MTC | Minimum Toxic Concentration |

When concentration is below MEC, efficacy may be insufficient.

When concentration is above MTC, toxicity risk may increase.

The appropriateness of a multiple-dosing regimen can be evaluated from the following perspectives:

- Is the concentration below MEC for a long time?
- Is the concentration above MTC for a long time?
- Is the steady-state trough concentration too low?
- Is the steady-state peak concentration too high?
- After adjusting Dose or $\tau$, do efficacy and safety improve?


In [ ]:
def calculate_time_in_ranges(t, concentration, mec, mtc, interval_start=None, interval_end=None):
    """
    Calculate time below MEC, within therapeutic window, and above MTC.
    """
    if interval_start is not None and interval_end is not None:
        mask = (t >= interval_start) & (t <= interval_end)
        t_use = t[mask]
        c_use = concentration[mask]
    else:
        t_use = t
        c_use = concentration

    if len(t_use) < 2:
        return np.nan, np.nan, np.nan

    dt = t_use[1] - t_use[0]
    time_below_mec = np.sum(c_use < mec) * dt
    time_within_window = np.sum((c_use >= mec) & (c_use <= mtc)) * dt
    time_above_mtc = np.sum(c_use > mtc) * dt

    return time_below_mec, time_within_window, time_above_mtc


def plot_multiple_dosing_therapeutic_window(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    tau_h=12,
    n_doses=10,
    ka_h=1.2,
    bioavailability=0.8,
    mec=2,
    mtc=12
):
    t_end_h = n_doses * tau_h
    t = np.linspace(0, t_end_h, 3000)

    concentration, dose_times = simulate_multiple_dosing(
        t=t,
        route=route,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        n_doses=n_doses,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    last_interval_start = (n_doses - 1) * tau_h
    last_interval_end = n_doses * tau_h

    total_below, total_within, total_above = calculate_time_in_ranges(
        t=t,
        concentration=concentration,
        mec=mec,
        mtc=mtc
    )

    last_below, last_within, last_above = calculate_time_in_ranges(
        t=t,
        concentration=concentration,
        mec=mec,
        mtc=mtc,
        interval_start=last_interval_start,
        interval_end=last_interval_end
    )

    metrics = calculate_interval_metrics(
        t=t,
        concentration=concentration,
        interval_start=last_interval_start,
        interval_end=last_interval_end
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, concentration, linewidth=2, label=f"{route}")
    ax.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax.fill_between(t, mec, mtc, alpha=0.15, label="Therapeutic window")

    for dose_time in dose_times:
        ax.axvline(dose_time, linestyle=":", alpha=0.25)

    ax.set_title("Multiple Dosing and Therapeutic Window")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Last-interval Cmax",
            "Last-interval Cmin",
            "Total time below MEC",
            "Total time within window",
            "Total time above MTC",
            "Last-interval time below MEC",
            "Last-interval time within window",
            "Last-interval time above MTC"
        ],
        "Value": [
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Cmin']:.2f} mg/L",
            f"{total_below:.2f} h",
            f"{total_within:.2f} h",
            f"{total_above:.2f} h",
            f"{last_below:.2f} h",
            f"{last_within:.2f} h",
            f"{last_above:.2f} h"
        ]
    })

    display(summary)


interact(
    plot_multiple_dosing_therapeutic_window,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    tau_h=FloatSlider(value=12, min=4, max=48, step=2, description="Tau"),
    n_doses=IntSlider(value=10, min=2, max=24, step=1, description="Doses"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=2, min=0.5, max=10, step=0.5, description="MEC"),
    mtc=FloatSlider(value=12, min=5, max=30, step=1, description="MTC")
);

## 13. Observation Task 3: Therapeutic-Window Analysis

Please use the simulation above to complete the following tasks.

### Task A: Standard regimen

Set:

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- tau = 12 h
- Doses = 10
- ka = 1.2 1/h
- F = 0.8
- MEC = 2 mg/L
- MTC = 12 mg/L

Record:

- Last-interval Cmax
- Last-interval Cmin
- Last-interval time below MEC
- Last-interval time above MTC

### Task B: Increase the dose

Change Dose to 1000 mg.

Observe:

- Does Cmax increase?
- Does Cmin increase?
- Does Time above MTC increase?
- Is increasing the dose always safer?

### Task C: Extend the dosing interval

Change tau to 24 h.

Observe:

- Does Cmin decrease?
- Does Time below MEC increase?
- Could the risk of insufficient efficacy increase?

### Task D: Decrease clearance

Change CL to 2 L/h.

Observe:

- Is the concentration more likely to exceed MTC?
- Is the time required to reach steady state prolonged?
- Does this suggest that the dosing regimen needs adjustment?


## 14. From Multiple-Dose PK to PD: Can Efficacy Be Sustained?

Notebook 1 introduced the Emax model:

$$
Effect = E_0 + \frac{E_{max} \cdot C^\gamma}{EC_{50}^{\gamma} + C^\gamma}
$$

During multiple dosing, plasma drug concentration fluctuates over time, so drug effect also fluctuates over time.

We can observe:

- Does the effect increase after each dose?
- Does the effect decrease near the end of the dosing interval?
- Does the average effect increase after multiple dosing?
- Does increasing the dose produce more effect, or does it mainly increase toxicity risk?
- Can shortening the dosing interval maintain a more stable effect?

The Effect in this section is expressed as a percentage and is used only for teaching simulation.


In [ ]:
def emax_effect(concentration, e0, emax, ec50, gamma):
    """
    Emax pharmacodynamic model.
    """
    concentration = np.asarray(concentration)
    effect = e0 + (emax * concentration**gamma) / (ec50**gamma + concentration**gamma)
    return effect


def plot_multiple_dosing_pkpd(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    tau_h=12,
    n_doses=10,
    ka_h=1.2,
    bioavailability=0.8,
    mec=2,
    mtc=12,
    emax=100,
    ec50=5,
    gamma=1.5
):
    t_end_h = n_doses * tau_h
    t = np.linspace(0, t_end_h, 3000)

    concentration, dose_times = simulate_multiple_dosing(
        t=t,
        route=route,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        n_doses=n_doses,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    effect = emax_effect(
        concentration=concentration,
        e0=0,
        emax=emax,
        ec50=ec50,
        gamma=gamma
    )

    last_interval_start = (n_doses - 1) * tau_h
    last_interval_end = n_doses * tau_h
    mask_last = (t >= last_interval_start) & (t <= last_interval_end)

    last_conc_metrics = calculate_interval_metrics(
        t=t,
        concentration=concentration,
        interval_start=last_interval_start,
        interval_end=last_interval_end
    )

    last_effect_metrics = calculate_interval_metrics(
        t=t,
        concentration=effect,
        interval_start=last_interval_start,
        interval_end=last_interval_end
    )

    average_effect_total = np.trapz(effect, t) / (t[-1] - t[0])
    average_effect_last = np.trapz(effect[mask_last], t[mask_last]) / (last_interval_end - last_interval_start)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(t, concentration, linewidth=2, label="Concentration")
    ax1.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax1.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax1.fill_between(t, mec, mtc, alpha=0.15, label="Therapeutic window")
    ax1.set_xlabel("Time (h)")
    ax1.set_ylabel("Concentration (mg/L)")

    ax2 = ax1.twinx()
    ax2.plot(t, effect, linewidth=2, linestyle="-.", label="Effect")
    ax2.set_ylabel("Effect (%)")

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

    ax1.set_title("Multiple Dosing PK/PD Simulation")
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Last-interval Cmax",
            "Last-interval Cmin",
            "Last-interval average concentration",
            "Last-interval maximum effect",
            "Last-interval minimum effect",
            "Total average effect",
            "Last-interval average effect"
        ],
        "Value": [
            f"{last_conc_metrics['Cmax']:.2f} mg/L",
            f"{last_conc_metrics['Cmin']:.2f} mg/L",
            f"{last_conc_metrics['Cavg_interval']:.2f} mg/L",
            f"{last_effect_metrics['Cmax']:.2f}%",
            f"{last_effect_metrics['Cmin']:.2f}%",
            f"{average_effect_total:.2f}%",
            f"{average_effect_last:.2f}%"
        ]
    })

    display(summary)


interact(
    plot_multiple_dosing_pkpd,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    tau_h=FloatSlider(value=12, min=4, max=48, step=2, description="Tau"),
    n_doses=IntSlider(value=10, min=2, max=24, step=1, description="Doses"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=2, min=0.5, max=10, step=0.5, description="MEC"),
    mtc=FloatSlider(value=12, min=5, max=30, step=1, description="MTC"),
    emax=FloatSlider(value=100, min=20, max=150, step=10, description="Emax"),
    ec50=FloatSlider(value=5, min=0.5, max=20, step=0.5, description="EC50"),
    gamma=FloatSlider(value=1.5, min=0.5, max=5, step=0.5, description="Hill")
);

## 15. Observation Task 4: PK/PD Relationship During Multiple Dosing

Please use the integrated PK/PD simulation to complete the following tasks.

### Task A: Standard regimen

Set:

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- tau = 12 h
- Doses = 10
- ka = 1.2 1/h
- F = 0.8
- MEC = 2 mg/L
- MTC = 12 mg/L
- Emax = 100
- EC50 = 5 mg/L
- Hill = 1.5

Record:

- Last-interval Cmax
- Last-interval Cmin
- Last-interval maximum effect
- Last-interval minimum effect
- Last-interval average effect

### Task B: Shorten the dosing interval

Change tau to 6 h.

Observe:

- Does Cmin increase?
- Does Minimum effect increase?
- Is the effect more stable?
- Could the risk of excessively high concentration increase?

### Task C: Increase the dose

Change Dose to 1000 mg.

Observe:

- Does Maximum effect increase noticeably?
- If the effect is already close to Emax, does the additional benefit of increasing the dose become smaller?
- Could Time above MTC increase?

### Task D: Decrease pharmacodynamic sensitivity

Change EC50 to 10 mg/L.

Observe:

- At the same concentration, does Effect decrease?
- Could higher exposure be required to achieve a similar effect?


## 16. Self-Assessment Questions: Multiple Dosing and Steady-State Concentration

Please complete the following self-assessment questions based on the content of this notebook. It is recommended that you answer independently first, and then check the reference answers in the next cell.

---

### Question 1: Which statement about steady-state concentration is the most accurate?

A. At steady state, plasma drug concentration no longer fluctuates at all  
B. At steady state, the concentration curve within each dosing interval is essentially repeated  
C. Steady state occurs only with IV dosing and not with oral dosing  
D. Steady state is determined only by dose and is unrelated to half-life  

---

### Question 2: Approximately how many half-lives are usually required to approach steady state?

A. 0.5 half-life  
B. 1 half-life  
C. 4–5 half-lives  
D. 20 half-lives  

---

### Question 3: When Dose and CL remain unchanged, what is the most likely result of shortening the dosing interval $\tau$?

A. Steady-state average concentration decreases  
B. Steady-state average concentration increases  
C. The drug is immediately and completely eliminated  
D. Half-life definitely becomes shorter  

---

### Question 4: During oral multiple dosing, what is the most direct result of decreased bioavailability F?

A. Steady-state average concentration decreases  
B. Clearance definitely increases  
C. Half-life definitely becomes longer  
D. The dosing interval automatically becomes shorter  

---

### Question 5: Which statement about Cmax and Cmin is correct?

A. Cmax is the lowest concentration within a dosing interval  
B. Cmin is the highest concentration within a dosing interval  
C. Cmax and Cmin can help evaluate peak-to-trough fluctuation and efficacy/toxicity risk  
D. During multiple dosing, Cmax and Cmin do not need to be considered


## 17. Reference Answers to the Self-Assessment Questions

### Question 1

**Reference answer: B**

**Explanation:**  
Steady state does not mean that plasma drug concentration no longer fluctuates at all. For intermittent dosing, steady state means that the concentration curve within each dosing interval is essentially repeated, including similar peak concentration, trough concentration, and AUCτ.

---

### Question 2

**Reference answer: C**

**Explanation:**  
Drugs with first-order elimination usually approach steady state after approximately 4–5 half-lives. The rate of reaching steady state is mainly determined by half-life, not by dose size.

---

### Question 3

**Reference answer: B**

**Explanation:**  
The steady-state average concentration can be expressed as:

For IV dosing:

$$
C_{ss,avg} = \frac{Dose}{CL \cdot \tau}
$$

For oral dosing:

$$
C_{ss,avg} = \frac{F \cdot Dose}{CL \cdot \tau}
$$

When Dose and CL remain unchanged, shortening $\tau$ increases the steady-state average concentration and may increase drug accumulation.

---

### Question 4

**Reference answer: A**

**Explanation:**  
During oral dosing:

$$
C_{ss,avg} = \frac{F \cdot Dose}{CL \cdot \tau}
$$

When Dose, CL, and $\tau$ remain unchanged, decreasing F reduces the amount of drug entering the systemic circulation, thereby decreasing the steady-state average concentration.

---

### Question 5

**Reference answer: C**

**Explanation:**  
Cmax is the highest concentration within the dosing interval, and Cmin is the lowest concentration within the dosing interval. Excessively high Cmax may indicate toxicity risk, while excessively low Cmin may indicate insufficient efficacy. Therefore, both are important metrics for evaluating multiple-dosing regimens.


## 18. Notebook Summary

This notebook simulated multiple dosing and steady-state concentration using a one-compartment model.

You should understand the following key conclusions:

1. Multiple-dose concentration profiles can be understood as the superposition of concentrations from repeated doses.
2. Drug accumulation occurs when a new dose is given before the previous dose has been sufficiently eliminated.
3. Steady state means that the concentration pattern within each dosing interval becomes repeatable, not that concentration is constant.
4. Half-life, dosing interval, clearance, and volume of distribution jointly determine accumulation, peak concentration, trough concentration, and average concentration.
5. Shorter dosing intervals may improve trough concentrations but can increase accumulation and toxicity risk.
6. Multiple-dose PK/PD analysis helps balance sustained efficacy and safety.

The complete logic of this section can be summarized as:

$$
Dose + \tau + CL + V_d \rightarrow Accumulation \rightarrow C_{ss} \rightarrow Effect(t) \rightarrow Efficacy/Safety
$$


